In [1]:
pip install tenseal scikit-learn numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 30.7 MB/s eta 0:00:00


## Imports

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import tenseal as ts

## Load Dataset (Simple Tabular)

In [3]:
data = load_diabetes()
X = data.data
y = data.target

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42
)

print("Dataset shape:", X.shape)

Dataset shape: (442, 10)


## Train Plain Linear Regression

In [7]:
plain_model = LinearRegression()
plain_model.fit(X_train, y_train)

LinearRegression()

In [8]:
y_pred_plain = plain_model.predict(X_test)

In [9]:
mse_plain = mean_squared_error(y_test, y_pred_plain)

print("\nPlaintext Model MSE:", mse_plain)


Plaintext Model MSE: 2900.193628493482


In [10]:
weights = plain_model.coef_
bias = plain_model.intercept_

## Setup Homomorphic Encryption (CKKS)

In [20]:
context = ts.context(
ts.SCHEME_TYPE.CKKS,
poly_modulus_degree=8192,
coeff_mod_bit_sizes=[60, 40, 40, 60]
)

context.global_scale = 2**40
context.generate_galois_keys()

## Encryption Function

In [21]:
def encrypt_vector(vec):
  return ts.ckks_vector(context, vec)

In [14]:
def encrypted_predict(enc_x, weights, bias):
  # dot product in encrypted domain
  result = enc_x.dot(weights)
  result += bias
  return result

## Run Encrypted Inference

In [25]:
encrypted_predictions = []

for i in range(len(X_test)):
    x = X_test[i]

    enc_x = encrypt_vector(x)
    enc_y = encrypted_predict(enc_x, weights, bias)

    y_hat = enc_y.decrypt()

    # Normalize output
    if isinstance(y_hat, (list, np.ndarray)):
        y_hat = y_hat[0]

    encrypted_predictions.append(y_hat)

encrypted_predictions = np.array(encrypted_predictions)

## Compare Results

In [26]:
mse_encrypted = mean_squared_error(y_test, encrypted_predictions)

print("\nEncrypted Model MSE:", mse_encrypted)


Encrypted Model MSE: 2900.1937341011276


## Compare Predictions

In [27]:
comparison_df = pd.DataFrame({
"Actual": y_test[:10],
"Plain_Pred": y_pred_plain[:10],
"Encrypted_Pred": encrypted_predictions[:10]
})

print("\nSample Predictions:")
print(comparison_df)


Sample Predictions:
   Actual  Plain_Pred  Encrypted_Pred
0   219.0  139.547558      139.547559
1    70.0  179.517208      179.517212
2   202.0  134.038756      134.038755
3   230.0  291.417029      291.417048
4   111.0  123.789659      123.789657
5    84.0   92.172347       92.172338
6   242.0  258.232389      258.232404
7   272.0  181.337321      181.337326
8    94.0   90.224113       90.224108
9    96.0  108.633759      108.633753


In [28]:
diff = np.abs(y_pred_plain - encrypted_predictions)

print("\nAverage Difference between Plain and Encrypted:",
np.mean(diff))


Average Difference between Plain and Encrypted: 6.4987903902048215e-06


In [29]:
print("\nSummary:")
print("Plain MSE :", mse_plain)
print("Encrypted MSE :", mse_encrypted)
print("Difference :", abs(mse_plain - mse_encrypted))


Summary:
Plain MSE : 2900.193628493482
Encrypted MSE : 2900.1937341011276
Difference : 0.00010560764576439396
